# **Data Understanding**

## Objectives

* Load the raw BankChurners file and find out what is actually in it before any
  cleaning decisions are made.
* Establish the size of the churn problem (BR1).
* Record every data quality issue that the cleaning notebook will have to deal
  with: disguised missing values, duplicate information, and columns that must
  not be used.

## Inputs

* `inputs/datasets/raw/BankChurners.csv`

## Outputs

* No data files. This notebook only inspects and reports.
* A profiling report saved to `outputs/reports/` for reference.
* A written list of issues, at the bottom, that notebook 02 acts on.

---

# Change working directory

The notebooks live in a subfolder, so the working directory has to be moved up
to the project root for the relative paths above to resolve.

In [1]:
import os
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main\\jupyter_notebooks'

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main'

---

# Section 1: Load the data

In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df = pd.read_csv('inputs/datasets/raw/BankChurners.csv')
df.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


The column names are readable and there is no obvious index column to set,
other than `CLIENTNUM` which is dealt with further down.

In [5]:
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    10,127
Columns: 23


10,127 customers and 23 columns. Small enough to hold in memory comfortably
and to plot without sampling.

---

# Section 2: Column types

In [6]:
df.dtypes.to_frame('dtype')

,dtype
CLIENTNUM,int64
Attrition_Flag,object
Customer_Age,int64
Gender,object
Dependent_count,int64
Education_Level,object
Marital_Status,object
Income_Category,object
Card_Category,object
Months_on_book,int64


Six columns are text and the rest are numeric. Two things to note:

* `Attrition_Flag` is the target and is currently text, not a 0/1 flag.
* `Dependent_count`, `Total_Relationship_Count`, `Months_Inactive_12_mon` and
  `Contacts_Count_12_mon` are stored as integers but are really small counts
  with a handful of levels each. They are worth treating as ordered categories
  when plotting.

In [7]:
df.select_dtypes('object').nunique().to_frame('unique values')

,unique values
Attrition_Flag,2
Gender,2
Education_Level,7
Marital_Status,4
Income_Category,6
Card_Category,4


---

# Section 3: The target variable (BR1)

BR1 asks how big the problem is and whether the two groups are balanced enough
to compare.

In [8]:
counts = df['Attrition_Flag'].value_counts()
pct = df['Attrition_Flag'].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({'customers': counts, 'percent': pct})

,customers,percent
Attrition_Flag,,
Existing Customer,8500,83.93
Attrited Customer,1627,16.07


**1,627 of 10,127 customers have churned - 16.07%.**

That answers BR1. The classes are imbalanced roughly 5:1, which has a direct
consequence for the rest of the analysis: raw counts will always be dominated by
the retained group, so every comparison from here on is made on **rates within
each group**, never on counts. A bar chart of raw counts would show that most
churners are Blue cardholders, but so is almost everybody, and it would say
nothing.

The imbalance is not severe enough to need resampling for exploratory work.
1,627 cases is plenty to characterise.

---

# Section 4: Missing values

In [9]:
df.isna().sum().to_frame('missing').query('missing > 0')

,missing


Nothing is missing in the sense pandas understands. That result is misleading,
so it is worth checking the categorical columns directly rather than trusting
it.

In [10]:
for col in df.select_dtypes('object').columns:
    print(f"=== {col} ===")
    print(df[col].value_counts().to_string())
    print()

=== Attrition_Flag ===
Attrition_Flag
Existing Customer    8500
Attrited Customer    1627

=== Gender ===
Gender
F    5358
M    4769

=== Education_Level ===
Education_Level
Graduate         3128
High School      2013
Unknown          1519
Uneducated       1487
College          1013
Post-Graduate     516
Doctorate         451

=== Marital_Status ===
Marital_Status
Married     4687
Single      3943
Unknown      749
Divorced     748

=== Income_Category ===
Income_Category
Less than $40K    3561
$40K - $60K       1790
$80K - $120K      1535
$60K - $80K       1402
Unknown           1112
$120K +            727

=== Card_Category ===
Card_Category
Blue        9436
Silver       555
Gold         116
Platinum      20



**The missing data is disguised as an `Unknown` category.**

| Column | Unknown | Share |
|---|---|---|
| `Education_Level` | 1,519 | 15.0% |
| `Income_Category` | 1,112 | 11.0% |
| `Marital_Status` | 749 | 7.4% |

This is the single most important thing to know before cleaning. `df.isna()`
reports a clean dataset and it is not one - roughly one customer in seven has no
recorded education level.

It also matters for the analysis itself. `Unknown` is not a value of education;
it is an absence of one. If it is left in and plotted as a seventh category, any
chart of education against churn silently invites the reader to compare
"Unknown" with "Graduate" as though they were comparable groups.

Whether these values are missing at random is checked in notebook 02, because
the answer decides how they are handled.

In [11]:
unknown_cols = ['Education_Level', 'Income_Category', 'Marital_Status']
any_unknown = (df[unknown_cols] == 'Unknown').any(axis=1)
print(f"Customers with at least one Unknown: {any_unknown.sum():,} "
      f"({any_unknown.mean():.1%})")
print(f"Customers with all three known:      {(~any_unknown).sum():,} "
      f"({(~any_unknown).mean():.1%})")

Customers with at least one Unknown: 3,046 (30.1%)
Customers with all three known:      7,081 (69.9%)


Nearly a third of the file has at least one Unknown. Dropping those rows would
throw away far too much, so they have to be handled rather than deleted.

---

# Section 5: Duplicates

In [12]:
print("Fully duplicated rows:      ", df.duplicated().sum())
print("Duplicated CLIENTNUM values:", df['CLIENTNUM'].duplicated().sum())

Fully duplicated rows:       0
Duplicated CLIENTNUM values: 0


No duplicate rows and no repeated customer IDs, so each row is one customer.

`CLIENTNUM` being unique across every row is exactly what makes it useless for
analysis - a column with 10,127 distinct values in 10,127 rows carries no
information about churn. It is dropped in notebook 02.

---

# Section 6: Numeric columns

In [13]:
numeric = df.select_dtypes(include=[np.number]).drop(columns=['CLIENTNUM'])
numeric.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Customer_Age,10127.0,46.33,8.02,26.0,41.00,46.00,52.00,73.00
Dependent_count,10127.0,2.35,1.30,0.0,1.00,2.00,3.00,5.00
Months_on_book,10127.0,35.93,7.99,13.0,31.00,36.00,40.00,56.00
Total_Relationship_Count,10127.0,3.81,1.55,1.0,3.00,4.00,5.00,6.00
Months_Inactive_12_mon,10127.0,2.34,1.01,0.0,2.00,2.00,3.00,6.00
Contacts_Count_12_mon,10127.0,2.46,1.11,0.0,2.00,2.00,3.00,6.00
Credit_Limit,10127.0,8631.95,9088.78,1438.3,2555.00,4549.00,11067.50,34516.00
Total_Revolving_Bal,10127.0,1162.81,814.99,0.0,359.00,1276.00,1784.00,2517.00
Avg_Open_To_Buy,10127.0,7469.14,9090.69,3.0,1324.50,3474.00,9859.00,34516.00
Total_Amt_Chng_Q4_Q1,10127.0,0.76,0.22,0.0,0.63,0.74,0.86,3.40


Some observations worth carrying forward:

* `Customer_Age` runs 26 to 73 with a mean of 46 - plausible, no impossible ages
  and no obvious sentinel values like 999.
* `Credit_Limit` is heavily right skewed: the median is 4,549 but the mean is
  8,632 and the maximum is 34,516. Skew this strong means the median is the
  honest summary, and plots will need care.
* `Total_Revolving_Bal` has a minimum of 0, and a sizeable number of customers
  sit exactly at 0. That is a real behaviour (paying the balance in full), not
  an error.
* `Total_Amt_Chng_Q4_Q1` and `Total_Ct_Chng_Q4_Q1` are ratios centred slightly
  below 1, meaning the typical customer spent a little less in Q4 than Q1.
* `Avg_Utilization_Ratio` runs the full 0 to 1 range as expected.

In [14]:
zero_balance = (df['Total_Revolving_Bal'] == 0).sum()
print(f"Customers with zero revolving balance: {zero_balance:,} "
      f"({zero_balance/len(df):.1%})")

Customers with zero revolving balance: 2,470 (24.4%)


---

# Section 7: Columns that duplicate other columns

Three of the credit columns look related. Rather than assume, the relationship
is tested directly.

In [15]:
implied = df['Credit_Limit'] - df['Total_Revolving_Bal']
diff = (implied - df['Avg_Open_To_Buy']).abs()
print("Avg_Open_To_Buy vs (Credit_Limit - Total_Revolving_Bal)")
print(f"  largest difference: {diff.max():.2e}")
print(f"  rows matching:      {(diff < 1e-6).sum():,} of {len(df):,}")

Avg_Open_To_Buy vs (Credit_Limit - Total_Revolving_Bal)
  largest difference: 5.68e-14
  rows matching:      10,127 of 10,127


In [16]:
implied_util = df['Total_Revolving_Bal'] / df['Credit_Limit']
diff_util = (implied_util - df['Avg_Utilization_Ratio']).abs()
print("Avg_Utilization_Ratio vs (Total_Revolving_Bal / Credit_Limit)")
print(f"  largest difference:   {diff_util.max():.4f}")
print(f"  rows matching to 3dp: {(diff_util < 1e-3).sum():,} of {len(df):,}")

Avg_Utilization_Ratio vs (Total_Revolving_Bal / Credit_Limit)
  largest difference:   0.0005
  rows matching to 3dp: 10,127 of 10,127


**Both are exactly derived, for all 10,127 rows.**

* `Avg_Open_To_Buy` = `Credit_Limit` - `Total_Revolving_Bal`
* `Avg_Utilization_Ratio` = `Total_Revolving_Bal` / `Credit_Limit`, rounded to 3dp

Neither carries any information the other three columns do not already hold. The
practical consequence is for interpretation: a correlation table will show
`Avg_Open_To_Buy` and `Credit_Limit` correlating at almost 1.0, and reporting
those as two separate findings would be double counting one fact.

`Avg_Utilization_Ratio` is kept despite being derived, because a ratio is
genuinely more interpretable to the retention team than the two raw figures it
comes from. `Avg_Open_To_Buy` is dropped in notebook 02.

---

# Section 8: The two Naive Bayes columns

The file ends with two columns whose names begin `Naive_Bayes_Classifier_...`.
The Kaggle dataset description says to delete them. It is worth confirming why
rather than taking it on trust.

In [17]:
nb_cols = [col for col in df.columns if col.startswith('Naive_Bayes')]
for col in nb_cols:
    print(col)
    print()
df[nb_cols].describe().T.round(4)

Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1

Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2



,count,mean,std,min,25%,50%,75%,max
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,10127.0,0.16,0.3653,0.0000,0.0001,0.0002,0.0003,0.9996
Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2,10127.0,0.84,0.3653,0.0004,0.9997,0.9998,0.9999,1.0000


In [18]:
churned = df['Attrition_Flag'] == 'Attrited Customer'
predicted = df[nb_cols[0]] > 0.5

print(f"Actual churn rate:            {churned.mean():.4f}")
print(f"Mean of first NB column:      {df[nb_cols[0]].mean():.4f}")
print(f"Agreement with actual target: {(predicted == churned).mean():.2%}")

Actual churn rate:            0.1607
Mean of first NB column:      0.1600
Agreement with actual target: 100.00%


**These columns are the answer.**

Thresholding the first one at 0.5 reproduces `Attrition_Flag` for every single
row - 100% agreement. They are the stored output of a classifier that was fitted
on the target, left in the file by mistake.

This is textbook target leakage. Any analysis including them would find them to
be overwhelmingly the strongest signal, and that finding would be worthless: it
would be the data telling us the answer we already gave it. They are dropped in
notebook 02 and play no part in the analysis.

---

# Section 9: Profiling report

A full profile is generated for reference. It is written straight to file
rather than displayed inline, which keeps this notebook small enough to review
on GitHub.

In [19]:
from ydata_profiling import ProfileReport

os.makedirs('outputs/reports', exist_ok=True)

profile = ProfileReport(df, title='BankChurners - raw data profile', minimal=True)
profile.to_file('outputs/reports/raw_data_profile.html')
print("Saved to outputs/reports/raw_data_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to outputs/reports/raw_data_profile.html


---

# Conclusions and Next Steps

**BR1 is answered.** 1,627 of 10,127 customers have churned, a rate of 16.07%.
The 5:1 imbalance means all later comparisons are made as rates within each
group rather than as counts.

**Data quality issues for notebook 02 to fix:**

1. `Education_Level`, `Income_Category` and `Marital_Status` use `Unknown` as a
   stand-in for missing, covering 15.0%, 11.0% and 7.4% of rows. Whether this is
   random needs testing before deciding how to treat it.
2. `CLIENTNUM` is unique per row and carries no information. Drop.
3. `Avg_Open_To_Buy` is exactly `Credit_Limit` - `Total_Revolving_Bal`. Drop as
   redundant.
4. The two `Naive_Bayes_Classifier_...` columns reproduce the target perfectly.
   Drop as leakage.
5. `Attrition_Flag` is text and needs a 0/1 version for rate calculations.

**Nothing wrong that was expected to be:** no duplicate rows, no impossible
values, no sentinel numbers, and every numeric range is plausible. The problems
in this file are the quiet kind - disguised missingness and columns that should
not be there - rather than the obvious kind.

Next: notebook 02, data cleaning.